# ReCAHS — Hierarchical Moving-Block Bootstrap Analysis

This notebook quantifies uncertainty in the five-seed ETTh1 pruning comparison while preserving the temporal dependence of overlapping forecast windows.

## Statistical design

- Paired losses are taken from the same test windows.
- Training seeds are resampled with replacement to represent model-training variability.
- Circular moving blocks are resampled within each selected seed to preserve local temporal dependence.
- The primary block length is **96 windows**, matching the forecast horizon.
- Block lengths **24, 96 and 168** are evaluated as a sensitivity analysis.
- Differences are defined as `left method − right method`; negative values favor the left method.
- A percentile 95% bootstrap confidence interval containing zero is reported as inconclusive.

The analysis does not retrain a model and does not require a GPU. It reads the `test_window_losses.csv` files generated by notebook 15.


In [ ]:
# 1) SETTINGS, DRIVE AND REPOSITORY
from pathlib import Path
from google.colab import drive
import subprocess

SEEDS = [7, 42, 1234, 2026, 3407]
PRIMARY_BLOCK_LENGTH = 96
SENSITIVITY_BLOCK_LENGTHS = [24, 96, 168]
BOOTSTRAP_REPLICATES = 10_000
BOOTSTRAP_RANDOM_SEED = 2026
BOOTSTRAP_CHUNK_SIZE = 250
CONFIDENCE_LEVEL = 0.95

REPO_URL = "https://github.com/didemneda/regime-aware-head-pruning.git"
FEATURE_BRANCH = "feat/reproducible-pipeline"
REPO_DIR = Path("/content/regime-aware-head-pruning")
DRIVE_MOUNT = Path("/content/drive")


def run(command, cwd=None, check=True):
    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    print("$", " ".join(map(str, command)))
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {command}")
    return result


if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))
else:
    print("Drive already mounted.")

PROJECT_DIR = DRIVE_MOUNT / "MyDrive" / "BIL401_Regime_Head_Pruning"
PRUNING_DIR = PROJECT_DIR / "multiseed/ETTh1/pruning"
DRIVE_OUTPUT_DIR = PRUNING_DIR / "block_bootstrap"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / ".git").exists():
    run([
        "git", "clone", "--branch", FEATURE_BRANCH,
        REPO_URL, REPO_DIR,
    ])

current_branch = run(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
).stdout.strip()
if current_branch != FEATURE_BRANCH:
    raise RuntimeError(
        f"Expected branch {FEATURE_BRANCH}, found {current_branch}."
    )

REPO_OUTPUT_DIR = REPO_DIR / "results/etth1/statistics"
REPO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Seeds:", SEEDS)
print("Primary block length:", PRIMARY_BLOCK_LENGTH)
print("Pruning inputs:", PRUNING_DIR)
print("Repository outputs:", REPO_OUTPUT_DIR)


In [ ]:
# 2) LOAD AND VALIDATE PAIRED WINDOW LOSSES
import numpy as np
import pandas as pd

METHOD_COLUMNS = {
    "unpruned_baseline": {"mse": "baseline_mse", "mae": "baseline_mae"},
    "static_25": {"mse": "static_mse", "mae": "static_mae"},
    "dynamic_joint_25": {
        "mse": "dynamic_joint_mse",
        "mae": "dynamic_joint_mae",
    },
}

COMPARISONS = [
    ("dynamic_joint_25", "static_25"),
    ("dynamic_joint_25", "unpruned_baseline"),
    ("static_25", "unpruned_baseline"),
]

REQUIRED_COLUMNS = {
    "window_id", "regime", "is_confident",
    "baseline_mse", "static_mse", "dynamic_joint_mse",
    "baseline_mae", "static_mae", "dynamic_joint_mae",
}

losses_by_seed = {}
expected_window_count = None
reference_window_ids = None
reference_regimes = None

for seed in SEEDS:
    path = PRUNING_DIR / f"seed_{seed}" / "test_window_losses.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing seed loss file: {path}")

    frame = pd.read_csv(path).sort_values("window_id").reset_index(drop=True)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"seed={seed}: missing columns {sorted(missing)}")

    numeric_columns = sorted(REQUIRED_COLUMNS - {"regime"})
    if frame[numeric_columns].isna().any().any():
        raise ValueError(f"seed={seed}: NaN values found")

    window_ids = frame["window_id"].to_numpy(dtype=int)
    regimes = frame["regime"].astype(str).to_numpy()
    if not np.array_equal(window_ids, np.arange(len(frame))):
        raise ValueError(f"seed={seed}: window_id is not sequential")

    if expected_window_count is None:
        expected_window_count = len(frame)
        reference_window_ids = window_ids
        reference_regimes = regimes
    else:
        if len(frame) != expected_window_count:
            raise ValueError(f"seed={seed}: inconsistent window count")
        if not np.array_equal(window_ids, reference_window_ids):
            raise ValueError(f"seed={seed}: window alignment mismatch")
        if not np.array_equal(regimes, reference_regimes):
            raise ValueError(f"seed={seed}: regime alignment mismatch")

    losses_by_seed[seed] = frame
    print(f"seed={seed}: {len(frame)} paired windows loaded")

print("Validated seeds:", len(losses_by_seed))
print("Windows per seed:", expected_window_count)
print("Paired seed-window observations:", len(SEEDS) * expected_window_count)


In [ ]:
# 3) HIERARCHICAL CIRCULAR MOVING-BLOCK BOOTSTRAP

def circular_block_means(values, block_length):
    """Return the mean of every circular block start for each seed."""
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError("values must have shape [seed, window]")

    seed_count, window_count = values.shape
    if not 1 <= block_length <= window_count:
        raise ValueError((block_length, window_count))

    result = np.empty((seed_count, window_count), dtype=np.float64)
    for seed_index in range(seed_count):
        row = values[seed_index]
        extended = np.concatenate([row, row[: block_length - 1]])
        cumulative = np.concatenate([[0.0], np.cumsum(extended)])
        block_sums = (
            cumulative[block_length : block_length + window_count]
            - cumulative[:window_count]
        )
        result[seed_index] = block_sums / block_length
    return result


def hierarchical_moving_block_bootstrap(
    paired_differences,
    block_length,
    replicates,
    random_seed,
    chunk_size=250,
):
    """Resample training seeds and circular temporal blocks."""
    paired_differences = np.asarray(paired_differences, dtype=np.float64)
    seed_count, window_count = paired_differences.shape
    block_means = circular_block_means(paired_differences, block_length)
    blocks_per_seed = int(np.ceil(window_count / block_length))
    rng = np.random.default_rng(random_seed)
    draws = np.empty(replicates, dtype=np.float64)

    for begin in range(0, replicates, chunk_size):
        end = min(begin + chunk_size, replicates)
        size = end - begin
        sampled_seeds = rng.integers(
            0, seed_count, size=(size, seed_count)
        )
        sampled_starts = rng.integers(
            0,
            window_count,
            size=(size, seed_count, blocks_per_seed),
        )
        selected = block_means[
            sampled_seeds[:, :, None],
            sampled_starts,
        ]
        draws[begin:end] = selected.mean(axis=(1, 2))
    return draws


def percentile_interval(draws, confidence_level=0.95):
    alpha = 1.0 - confidence_level
    return tuple(
        np.quantile(draws, [alpha / 2.0, 1.0 - alpha / 2.0])
    )


# Deterministic implementation self-test.
synthetic = np.tile(np.arange(20, dtype=float), (3, 1))
synthetic_draws = hierarchical_moving_block_bootstrap(
    synthetic,
    block_length=4,
    replicates=100,
    random_seed=1,
    chunk_size=25,
)
assert synthetic_draws.shape == (100,)
assert np.isfinite(synthetic_draws).all()
print("Bootstrap implementation self-test passed.")


In [ ]:
# 4) PRIMARY ANALYSIS AND BLOCK-LENGTH SENSITIVITY
sensitivity_rows = []
per_seed_rows = []

for metric_index, metric in enumerate(["mse", "mae"]):
    matrices = {
        method: np.stack([
            losses_by_seed[seed][columns[metric]].to_numpy(dtype=np.float64)
            for seed in SEEDS
        ])
        for method, columns in METHOD_COLUMNS.items()
    }

    for comparison_index, (left_method, right_method) in enumerate(COMPARISONS):
        differences = matrices[left_method] - matrices[right_method]
        observed = float(differences.mean())
        reference_mean = float(matrices[right_method].mean())
        relative_percent = 100.0 * observed / reference_mean

        for seed_index, seed in enumerate(SEEDS):
            seed_difference = float(differences[seed_index].mean())
            seed_reference = float(matrices[right_method][seed_index].mean())
            per_seed_rows.append({
                "seed": seed,
                "metric": metric,
                "left_method": left_method,
                "right_method": right_method,
                "mean_difference": seed_difference,
                "relative_difference_percent": (
                    100.0 * seed_difference / seed_reference
                ),
            })

        for block_length in SENSITIVITY_BLOCK_LENGTHS:
            derived_seed = (
                BOOTSTRAP_RANDOM_SEED
                + metric_index * 100_000
                + comparison_index * 10_000
                + block_length
            )
            draws = hierarchical_moving_block_bootstrap(
                differences,
                block_length=block_length,
                replicates=BOOTSTRAP_REPLICATES,
                random_seed=derived_seed,
                chunk_size=BOOTSTRAP_CHUNK_SIZE,
            )
            ci_low, ci_high = percentile_interval(draws, CONFIDENCE_LEVEL)

            if ci_high < 0:
                conclusion = "left_better"
            elif ci_low > 0:
                conclusion = "right_better"
            else:
                conclusion = "inconclusive"

            sensitivity_rows.append({
                "metric": metric,
                "left_method": left_method,
                "right_method": right_method,
                "difference_definition": "left_minus_right",
                "block_length": block_length,
                "seed_count": len(SEEDS),
                "windows_per_seed": expected_window_count,
                "bootstrap_replicates": BOOTSTRAP_REPLICATES,
                "observed_mean_difference": observed,
                "relative_difference_percent": relative_percent,
                "bootstrap_standard_error": float(draws.std(ddof=1)),
                "ci_level": CONFIDENCE_LEVEL,
                "ci_low": float(ci_low),
                "ci_high": float(ci_high),
                "probability_left_better": float(np.mean(draws < 0)),
                "ci_excludes_zero": bool(ci_low > 0 or ci_high < 0),
                "conclusion": conclusion,
            })

sensitivity_df = pd.DataFrame(sensitivity_rows)
per_seed_effects_df = pd.DataFrame(per_seed_rows)
primary_df = (
    sensitivity_df[
        sensitivity_df["block_length"] == PRIMARY_BLOCK_LENGTH
    ]
    .sort_values(["metric", "left_method", "right_method"])
    .reset_index(drop=True)
)

robustness_df = (
    sensitivity_df
    .groupby(["metric", "left_method", "right_method"], as_index=False)
    .agg(
        block_lengths_tested=("block_length", "nunique"),
        distinct_conclusions=("conclusion", "nunique"),
        all_ci_exclude_zero=("ci_excludes_zero", "all"),
        min_ci_low=("ci_low", "min"),
        max_ci_high=("ci_high", "max"),
    )
)
robustness_df["robust_to_block_length"] = (
    robustness_df["distinct_conclusions"] == 1
)

print("Primary 96-window results")
display(primary_df[[
    "metric", "left_method", "right_method",
    "observed_mean_difference", "relative_difference_percent",
    "ci_low", "ci_high", "probability_left_better", "conclusion",
]])

print("Block-length robustness")
display(robustness_df)


In [ ]:
# 5) SAVE TO DRIVE AND REPOSITORY
import json

outputs = {
    "block_bootstrap_pairwise.csv": primary_df,
    "block_bootstrap_sensitivity.csv": sensitivity_df,
    "block_bootstrap_per_seed_effects.csv": per_seed_effects_df,
    "block_bootstrap_robustness.csv": robustness_df,
}

for filename, dataframe in outputs.items():
    dataframe.to_csv(DRIVE_OUTPUT_DIR / filename, index=False)
    dataframe.to_csv(REPO_OUTPUT_DIR / filename, index=False)
    print("Saved:", REPO_OUTPUT_DIR / filename)

metadata = {
    "analysis": "hierarchical_circular_moving_block_bootstrap",
    "difference_definition": "left_method_minus_right_method",
    "seeds": SEEDS,
    "primary_block_length": PRIMARY_BLOCK_LENGTH,
    "sensitivity_block_lengths": SENSITIVITY_BLOCK_LENGTHS,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_random_seed": BOOTSTRAP_RANDOM_SEED,
    "confidence_level": CONFIDENCE_LEVEL,
    "windows_per_seed": expected_window_count,
    "resample_training_seeds": True,
    "resample_temporal_blocks_within_seed": True,
}

for directory in [DRIVE_OUTPUT_DIR, REPO_OUTPUT_DIR]:
    (directory / "block_bootstrap_metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n",
        encoding="utf-8",
    )

summary_lines = [
    "# ETTh1 Hierarchical Moving-Block Bootstrap Summary",
    "",
    "Differences are `left method - right method`; negative values favor the left method.",
    "The primary analysis resamples training seeds and circular blocks of 96 windows.",
    "",
    "| Metric | Left | Right | Difference | 95% CI | Conclusion |",
    "|---|---|---|---:|---:|---|",
]

for row in primary_df.itertuples(index=False):
    summary_lines.append(
        f"| {row.metric.upper()} | {row.left_method} | {row.right_method} "
        f"| {row.observed_mean_difference:.8f} "
        f"| [{row.ci_low:.8f}, {row.ci_high:.8f}] "
        f"| {row.conclusion} |"
    )

summary_text = "\n".join(summary_lines) + "\n"
for directory in [DRIVE_OUTPUT_DIR, REPO_OUTPUT_DIR]:
    (directory / "block_bootstrap_summary.md").write_text(
        summary_text,
        encoding="utf-8",
    )

print("\nPrimary result table:")
display(primary_df)

print("\nRepository changes:")
run(["git", "status", "--short"], cwd=REPO_DIR)

print(
    "\nDo not commit or push yet. Send the primary result table "
    "and robustness table for interpretation."
)
